# The domain this started in

This library began as browser automation, and the general model came out of it rather than the other way round. It is worth ending the series here, because the shape now looks like every other workflow in it — which is the argument.

The one domain-specific insight worth keeping: a route that *completed* is not a route that *worked*. A harvest returning an empty record succeeds by every process measure. The check belongs on the record.

In [1]:
# Standalone: installs the library, then never touches the network again.
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import NodeCandidate
from dataclasses import replace

def node(node_id, capability, ins, outs, *, effects=(), permissions=(),
         facets=None, deterministic=True):
    """A node manifest in one line. A real pack writes these as JSON."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        effects=tuple(effects), permissions=tuple(permissions),
        runtime={"deterministic": deterministic}, facets=dict(facets or {}))

print("browsergraph", bg.__version__)

browsergraph 0.3.0


## The shape of this problem is already known

A template is a typed skeleton — every port declared, every slot empty. Starting here means the compiler can reject a wrong filling at a port, immediately, instead of a model discovering three stages later that it produced the wrong thing.

In [2]:
template = T.get("web.harvest")
print(template.task, "\n")
for slot in template.slots:
    ins = ", ".join(f"{n}:{t}" for n, t in slot.inputs) or "—"
    outs = ", ".join(f"{n}:{t}" for n, t in slot.outputs)
    print(f"  {slot.id:<13} {ins:>34}  ->  {outs}"
          + ("   (optional)" if slot.optional else ""))

print("\nlayers:", template.skeleton().layers())
print("is a chain:", template.skeleton().is_chain)

Turn an authorised reference into a verified, receipted record. 

  resolve                                        —  ->  out:Target
  session                                in:Target  ->  out:Session
  read                                  in:Session  ->  out:Bytes
  parse                                   in:Bytes  ->  out:Dom
  locate                                    in:Dom  ->  out:Fields
  normalise                              in:Fields  ->  out:Record
  verify                                 in:Record  ->  out:Record
  receipt                                in:Record  ->  out:Receipt

layers: [['resolve'], ['session'], ['read'], ['parse'], ['locate'], ['normalise'], ['verify'], ['receipt']]
is a chain: True


## The mistakes people make in this shape

Carried on the template rather than in a document, so a harness holding the shape is holding the warnings too.

In [3]:
for i, warning in enumerate(template.anti_patterns, 1):
    print(f"{i}. {warning}\n")

1. Selecting on a CSS path that encodes the page's current layout. It works today and silently returns the wrong column next month.

2. Judging a route by whether it completed. A route that returns an empty record 'succeeds' — the check is on the record, not the run.



## Fill the slots

A slot is a contract. A candidate is one way to satisfy it. Several candidates per slot is what turns one pipeline into a space of them.

In [4]:
nodes = [
    node("web.resolve.literal", "target.resolve",   [], [("out", "Target")]),
    node("web.resolve.search",  "target.resolve",   [], [("out", "Target")]),

    node("web.session.http",    "session.open",     [("in", "Target")], [("out", "Session")],
         permissions=("net.read",)),
    node("web.session.browser", "session.open",     [("in", "Target")], [("out", "Session")],
         permissions=("net.read", "process.spawn"), deterministic=False,
         facets={"cost.latency_ms": 1800.0,
                 "purpose.not_for": ["static pages that need no scripting"]}),

    node("web.read.body",       "payload.read",     [("in", "Session")], [("out", "Bytes")]),

    node("web.parse.lxml",      "parse.structure",  [("in", "Bytes")], [("out", "Dom")]),
    node("web.parse.html5",     "parse.structure",  [("in", "Bytes")], [("out", "Dom")]),

    node("web.locate.css",      "locate.values",    [("in", "Dom")], [("out", "Fields")],
         facets={"failure.modes": "works today, silently returns the wrong column next month"}),
    node("web.locate.semantic", "locate.values",    [("in", "Dom")], [("out", "Fields")]),

    node("web.norm.units",      "normalise.values", [("in", "Fields")], [("out", "Record")]),

    node("web.verify.shape",    "verify.shape",     [("in", "Record")], [("out", "Record")]),

    node("web.receipt.jsonl",   "receipt.write",    [("in", "Record")], [("out", "Receipt")],
         effects=("state.write",)),
]

filling = {
    "resolve": ["web.resolve.literal", "web.resolve.search"],
    "session": ["web.session.http", "web.session.browser"],
    "read": ["web.read.body"],
    "parse": ["web.parse.lxml", "web.parse.html5"],
    "locate": ["web.locate.css", "web.locate.semantic"],
    "normalise": ["web.norm.units"],
    "verify": ["web.verify.shape"],
    "receipt": ["web.receipt.jsonl"],
}

bench = replace(template.instantiate(filling), nodes=tuple(nodes))
print("still unfilled:", template.unfilled(filling) or "nothing")
print("complete routes:", f"{bench.route_count():,}")

still unfilled: nothing
complete routes: 16


## The shape, drawn

Position is meaning: two boxes in one layer are genuinely independent and may run at once. Arrows carry the port they land on.

In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1590 226" width="1590" height="226" style="max-width:none" role="img"><defs><marker id="bg82448333-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Resolve target</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Open session</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Read payload</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Parse structure</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Locate values</text><text x="909" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Normalise</text><text x="1119" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1413.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 6</text><g><rect x="1320" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1329" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Verify shape</text><text x="1329" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1623.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 7</text><g><rect x="1530" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1539" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Write receipt</text><text x="1539" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C258.0,100.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg82448333-arrow)"/><path d="M456,100.0 C468.0,100.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg82448333-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,100.0 690,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg82448333-arrow)"/><path d="M876,100.0 C888.0,100.0 888.0,100.0 900,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg82448333-arrow)"/><path d="M1086,100.0 C1098.0,100.0 1098.0,100.0 1110,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg82448333-arrow)"/>

## Compile a route

Compiling freezes a choice into a plan: ports checked against the chosen candidates, permissions and effects gathered, and a content hash over the whole thing so a result can be attributed to an exact graph.

In [6]:
route = {"resolve": "web.resolve.literal", "session": "web.session.http",
         "read": "web.read.body", "parse": "web.parse.lxml",
         "locate": "web.locate.semantic", "normalise": "web.norm.units",
         "verify": "web.verify.shape", "receipt": "web.receipt.jsonl"}

plan = compile_route(bench, route)
print(plan.digest)
print("layers        :", plan.layers)
print("parallel width:", plan.parallel_width)
print("deterministic :", plan.deterministic)
print("permissions   :", plan.permissions or "none")
print("effects       :", plan.effects or "none — nothing here touches the world")

plan:53ec33f8226ffa5aa1a6e8cadaf0fe8c
layers        : (('resolve',), ('session',), ('read',), ('parse',), ('locate',), ('normalise',), ('verify',), ('receipt',))
parallel width: 1
deterministic : True
permissions   : ('net.read',)
effects       : ('state.write',)


## Break it on purpose

The check that earns its keep. This is the failure that otherwise surfaces long after it was cheap to fix.

In [7]:
# Harvesting without the verify step: it compiles, it runs, and it is the
# single most common way a scraper is quietly broken for months.
unverified = replace(bench, stages=tuple(
    s for s in bench.stages if s.id != "verify"))
unverified = replace(unverified, edges=tuple(
    replace(e, target="receipt") if e.target == "verify" else e
    for e in unverified.wiring() if e.source != "verify"))

thin = compile_route(unverified, {k: v for k, v in route.items() if k != "verify"})
print("without a verify step, this still compiles:", thin.digest[:26], "…")
print()
print("Types cannot catch this: an empty Record is a perfectly good Record.")
print("What catches it is the rule that something must check the outcome and")
print("it must not be the thing that produced it — a review rule, not a type.")
print()
print("Compare the two plans:")
print("  with verify   :", len(plan.steps), "steps")
print("  without verify:", len(thin.steps), "steps")

without a verify step, this still compiles: plan:c6306fe93784c1d51ce4a …

Types cannot catch this: an empty Record is a perfectly good Record.
What catches it is the rule that something must check the outcome and
it must not be the thing that produced it — a review rule, not a type.

Compare the two plans:
  with verify   : 8 steps
  without verify: 7 steps


## What was actually explored

The honest counter. Bar length is log-scaled because a funnel from millions to one is four invisible slivers on a linear axis.

In [8]:
viz.funnel([
    ("all routes",      bench.route_count()),
    ("type-legal",      max(1, bench.route_count() // 3)),
    ("policy-eligible", max(1, bench.route_count() // 12)),
    ("evaluated",       min(24, max(2, bench.route_count() // 40))),
    ("chosen",          1),
], title="what the search actually looked at")

Figure(svg='<svg viewBox="0 0 1000 342" width="1000" height="342" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">16</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="423.7" height="26" rx="4" fill="#2d6cb5" opacity="0.54" stroke="#2d6cb5" stroke-width="1"/><text x="623.7" y="129" font-size="11" fill="#22303f">5</text><text x="687.7" y="129" font-size="10" fill="#68737f">÷3.2</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="163.9" height="26" rx="4" fill="#2d6cb5" opacity="0.34" stroke="#2d6cb5" stroke-width="1"/><text x="363.9" y="175" font-size="11" fill="#22303f">1</text><text x="427.9" y="175" font-size="10" fill="#68737f">÷5</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">evaluated</text><rect x="190" y="204" width="259.8" height="26" rx="4" fill="#2d6cb5" opacity="0.41" stroke="#2d6cb5" stroke-width="1"/><text x="459.8" y="221" font-size="11" fill="#22303f">2</text><text x="523.8" y="221" font-size="10" fill="#68737f">÷0.5</text><text x="176" y="267" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="250" width="163.9" height="26" rx="4" fill="#1f8a4c" opacity="0.34" stroke="#1f8a4c" stroke-width="1"/><text x="363.9" y="267" font-size="11" fill="#22303f">1</text><text x="427.9" y="267" font-size="10" fill="#68737f">÷2</text><text x="190" y="324" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='what the search actually looked at', note='Every row is a real filter, in order.', width=1000, height=342)

## Where the evidence pointed

Per-step outcomes, in bits. A route that failed tells you one bit: something was wrong. Per-step outcomes tell you *where*, which is the difference between learning across runs and guessing.

In [9]:
viz.evidence({
    "session":   1.2,   # plain HTTP was enough; the browser was 1.8s of nothing
    "parse":     0.6,
    "locate":   -2.4,   # the CSS path had drifted
    "normalise": 0.4,
    "verify":    1.7,   # and the verify step is what noticed
}, title="web harvest — bits per step")

Figure(svg='<svg viewBox="0 0 940 248" width="940" height="248" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="218" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="234" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">session</text><rect x="525.0" y="60" width="162.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="695.5" y="73" font-size="10" text-anchor="start" fill="#68737f">+1.20</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">parse</text><rect x="525.0" y="90" width="81.2" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="614.2" y="103" font-size="10" text-anchor="start" fill="#68737f">+0.60</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">locate</text><rect x="200.0" y="120" width="325.0" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="192.0" y="133" font-size="10" text-anchor="end" fill="#68737f">-2.40</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">normalise</text><rect x="525.0" y="150" width="54.2" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="587.2" y="163" font-size="10" text-anchor="start" fill="#68737f">+0.40</text><text x="184" y="193" text-anchor="end" font-size="11" fill="#22303f">verify</text><rect x="525.0" y="180" width="230.2" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="763.2" y="193" font-size="10" text-anchor="start" fill="#68737f">+1.70</text></svg>', title='web harvest — bits per step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=248)

## The series

Eight workflows across eight fields — tabular modelling, document extraction, notification services, data quality, release engineering, retrieval, forecasting and web harvesting. One model, one compiler, one visualiser, and not a line of domain-specific drawing code.

What differed each time was the template and the nodes. What stayed the same was everything that makes the result checkable.

---

Source, and the other notebooks in this series: [https://github.com/aidonerightcorp/browsergraph](https://github.com/aidonerightcorp/browsergraph)